<a href="https://colab.research.google.com/github/tmu-wsi-mil-project/attention-mil/blob/main/attention_mil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Path where you uploaded the dataset
DATASET_PATH = '/content/drive/MyDrive/SICAPv2'

Mounted at /content/drive


In [11]:
import os
print("DATASET_PATH:", DATASET_PATH)

print("\nTop-level contents:")
for item in os.listdir(DATASET_PATH):
    print(" -", item)


DATASET_PATH: /content/drive/MyDrive/SICAPv2

Top-level contents:
 - wsi_labels.xlsx
 - readme.txt
 - masks
 - partition
 - images


Deeper Peek into the dataset

In [12]:
for root, dirs, files in os.walk(DATASET_PATH):
    print("ROOT:", root)
    print("DIRS:", dirs[:10])
    print("FILES:", files[:10])
    print("---------------")
    break   # only top level


ROOT: /content/drive/MyDrive/SICAPv2
DIRS: ['masks', 'partition', 'images']
FILES: ['wsi_labels.xlsx', 'readme.txt']
---------------


In [13]:
ROOT = "/content/drive/MyDrive/SICAPv2"
PATCH_IMAGES = os.path.join(ROOT, "images")
LABELS_XLSX = os.path.join(ROOT, "wsi_labels.xlsx")

print("Images dir exists:", os.path.isdir(PATCH_IMAGES))
print("Labels file exists:", os.path.isfile(LABELS_XLSX))

Images dir exists: True
Labels file exists: True


This labels tell us the slide level labels not the patch level

In [14]:
import pandas as pd

labels_df = pd.read_excel(LABELS_XLSX)
print("Columns in labels_df:")
print(labels_df.columns.tolist())
labels_df.head()


Columns in labels_df:
['slide_id', 'patient_id', 'Gleason_primary', 'Gleason_secondary']


,slide_id,patient_id,Gleason_primary,Gleason_secondary
0,16B0001851,667360,4,5
1,16B0003388,325687,4,4
2,16B0003394,747184,3,3
3,16B0006668,14107,5,5
4,16B0006669,14107,5,5


Gleason_primary indicates the most common label on the slide.
Gleason_secondary is the second most common label on the slide.

1 = high-grade slide (any presence of Gleason 4 or 5)
0 = low-grade slide (only Gleason 3 or benign 0)

In [15]:
labels_df["weak_label"] = (
    (labels_df["Gleason_primary"] >= 4) |
    (labels_df["Gleason_secondary"] >= 4)
).astype(int)

labels_df[["slide_id", "Gleason_primary", "Gleason_secondary", "weak_label"]].head()


,slide_id,Gleason_primary,Gleason_secondary,weak_label
0,16B0001851,4,5,1
1,16B0003388,4,4,1
2,16B0003394,3,3,0
3,16B0006668,5,5,1
4,16B0006669,5,5,1


The labels excels say that somewhere in this slide there are high-grade cancer patterns (4+5), but we don't know which patch contains the high-grade gland, low-grade gland, and benign.

And how does it learn?
By analyzing each patch.

Glob find all patch images in the images folder that ends with .jpg and .png

In [16]:
import glob

patch_paths = glob.glob(os.path.join(PATCH_IMAGES, "*.jpg")) + \
              glob.glob(os.path.join(PATCH_IMAGES, "*.png"))

print("Total patches:", len(patch_paths))
print("Example patch:", patch_paths[0])


Total patches: 18783
Example patch: /content/drive/MyDrive/SICAPv2/images/18B0006179I_Block_Region_1_2_4_xini_15824_yini_73505.jpg


Extract slide_id from each patch filename - (before first _ underscore)

In [17]:
def extract_slide_id(fname):
    base = os.path.basename(fname)
    name = os.path.splitext(base)[0]

    # BEST RULE for your dataset:
    # slide_id = everything before the FIRST underscore
    return name.split("_")[0]

In [18]:
example = patch_paths[0]
print(example)
print("Extracted slide_id:", extract_slide_id(example))


/content/drive/MyDrive/SICAPv2/images/18B0006179I_Block_Region_1_2_4_xini_15824_yini_73505.jpg
Extracted slide_id: 18B0006179I


Join slide labels with patch file names - patch_path is required to be the absolute path.

In [19]:
# slide_id → weak label dictionary
slide_to_label = dict(zip(labels_df["slide_id"].astype(str),
                          labels_df["weak_label"]))

patch_records = []

for path in patch_paths:
    slide_id = extract_slide_id(path)

    # ensure slide_id exists in labels file
    if slide_id not in slide_to_label:
        continue

    patch_records.append({
        "patch_path": path,
        "slide_id": slide_id,
        "weak_label": slide_to_label[slide_id]
    })

patch_df = pd.DataFrame(patch_records)
patch_df.head()


,patch_path,slide_id,weak_label
0,/content/drive/MyDrive/SICAPv2/images/18B00061...,18B0006179I,1
1,/content/drive/MyDrive/SICAPv2/images/18B00061...,18B0006179I,1
2,/content/drive/MyDrive/SICAPv2/images/18B00061...,18B0006179I,1
3,/content/drive/MyDrive/SICAPv2/images/18B00061...,18B0006179I,1
4,/content/drive/MyDrive/SICAPv2/images/18B00061...,18B0006179I,1


In [20]:
import pandas as pd

# If patch_df is already in memory, you don't need this line.
# Otherwise, load it from CSV if you saved it earlier:
# patch_df = pd.read_csv("/content/drive/MyDrive/SICAPv2/patch_weak_labels.csv")

patch_df.head()
print("Number of rows (patches):", len(patch_df))


Number of rows (patches): 18783


In [24]:
bags = []

# group all patches by slide
for slide_id, group in patch_df.groupby("slide_id"):
    patch_paths = group["patch_path"].tolist()   # all patches for this slide
    label = int(group["weak_label"].iloc[0])     # weak label for this slide

    if len(patch_paths) == 0:
        continue  # skip empty bags just to be safe

    bag = {
        "slide_id": slide_id,        # e.g. '18B0002930B'
        "patch_paths": patch_paths,  # list of all patch image paths for that slide
        "label": label,              # 0 or 1 (your weak_label)
    }
    bags.append(bag)

print("Total MIL bags (slides):", len(bags))
print("Example bag slide_id:", bags[0]["slide_id"])
print("Num patches in this bag:", len(bags[1]["patch_paths"]))
print("Label for this bag:", bags[0]["label"])


Total MIL bags (slides): 155
Example bag slide_id: 16B0001851
Num patches in this bag: 114
Label for this bag: 1


In [30]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

# basic transforms
patch_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class MILBagsDataset(Dataset):
    def __init__(self, bags, transform=None, max_patches_per_bag=512):
        """
        bags: list of dicts with keys:
              'slide_id', 'patch_paths', 'label'
        """
        self.bags = bags
        self.transform = transform
        self.max_patches_per_bag = max_patches_per_bag

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag_info = self.bags[idx]
        slide_id = bag_info["slide_id"]
        patch_paths = bag_info["patch_paths"]
        label = bag_info["label"]

        # (optional) randomly drop patches if too many
        if len(patch_paths) > self.max_patches_per_bag:
            patch_paths = random.sample(patch_paths, self.max_patches_per_bag)

        imgs = []
        for p in patch_paths:
            img = Image.open(p).convert("RGB")
            if self.transform:
                img = self.transform(img)
            imgs.append(img)

        # bag tensor: [num_patches, 3, H, W]
        bag_tensor = torch.stack(imgs, dim=0)
        label_tensor = torch.tensor(label, dtype=torch.long)

        return bag_tensor, label_tensor, slide_id


In [28]:
def mil_collate_fn(batch):
    bag_tensors, labels, slide_ids = zip(*batch)
    return list(bag_tensors), torch.stack(labels), list(slide_ids)


In [29]:
import models
base = models.resnet18(pretrained=True)
self.feature_extractor = nn.Sequential(*list(base.children())[:-1])  # drop final FC
self.feat_dim = 512


NameError: name 'models' is not defined